# nb10.2 - Causal Forest on Synthetic Data: Where the Effect Is Not the Average

*Week 10, Chapter 10.3 companion. Sam wants to know whether his momentum trading strategy works the
same way for small-cap and large-cap stocks - and whether the answer changes across volatility
regimes. The average treatment effect (ATE) reported in his paper is a single number. The discussant
asks: "Where does that average come from? Show me the heterogeneity." This notebook builds the
machinery he needs.*

The reveal-the-trick lesson: **a causal forest estimates a different treatment effect for each
covariate cell, $\widehat\tau(\mathbf{x})$, by training a forest of trees that split on covariates to
maximize heterogeneity in the treatment effect rather than to predict the outcome.** Two design
ingredients keep the estimator honest:

1. **Honest splitting (Athey & Imbens 2016, *PNAS* 113(27):7353-7360).** Use one half of the sample to
   decide the splits, the other half to compute leaf estimates. Without this, the leaves are
   overfit and the confidence intervals lie.
2. **Best linear projection (Semenova & Chernozhukov 2021, *Econometrics Journal* 24(2):264-289).**
   Project $\widehat\tau(\mathbf{x})$ onto a small linear function of summary covariates. Reports a
   handful of interpretable slopes, with valid standard errors, rather than asking a referee to stare
   at thousands of leaf values.

EconML's official `CausalForestDML` ships these defaults. EconML is not on the pinned stack for the
camp environment, so this notebook **hand-rolls** the same logic on top of scikit-learn's
`DecisionTreeRegressor` and `RandomForestRegressor`. The pedagogy is the point: by the end you can
explain - in code - what `CausalForestDML` is actually doing under the hood.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.3f}")

## 1. A heterogeneous-effects DGP

Each unit $i$ has covariates $\mathbf{x}_i = (x_{i1}, x_{i2}, x_{i3}, x_{i4})$ drawn iid from
$\mathcal{N}(0, I_4)$. Two of the four covariates are *effect modifiers*, the other two are noise.
The treatment is randomized with propensity $0.5$, so we can compute the truth without worrying about
selection. The conditional average treatment effect is

$$\tau(\mathbf{x}) = 1.0 + 1.5 \cdot \mathbb{1}\{x_1 > 0\} - 0.8 \cdot x_2.$$

Reading the formula: there is a baseline effect of $1.0$, an extra $1.5$ for units with $x_1 > 0$ (a
discrete jump), and a linear damp $-0.8 x_2$. Units with negative $x_1$ and large positive $x_2$ have
near-zero or even negative treatment effects, while units with positive $x_1$ and negative $x_2$ get
the biggest gains. Variables $x_3$ and $x_4$ are noise. The outcome equation adds a baseline
$g(\mathbf{x}) = 0.7 x_1 + 0.3 x_3$ plus normal noise.

In [ ]:
def make_cate_dgp(rng, N=2000):
    X = rng.normal(0, 1, size=(N, 4))
    D = rng.binomial(1, 0.5, size=N)              # randomized treatment
    tau_x = 1.0 + 1.5 * (X[:, 0] > 0) - 0.8 * X[:, 1]
    g_x = 0.7 * X[:, 0] + 0.3 * X[:, 2]
    eps = rng.normal(0, 1.0, size=N)
    Y = g_x + tau_x * D + eps
    df = pd.DataFrame(X, columns=[f"x{j+1}" for j in range(4)])
    df["D"] = D; df["Y"] = Y; df["tau_true"] = tau_x
    return df

df = make_cate_dgp(rng)
print(df.shape)
print("ATE (truth) =", df["tau_true"].mean().round(3))
print("range of true tau:", (df["tau_true"].min().round(2), df["tau_true"].max().round(2)))
df.head()

**Sanity check.** The ATE is the *average* of $\tau(\mathbf{x})$. The range tells us the
heterogeneity is real: $\tau$ varies from roughly $-1.5$ (someone with $x_1 \le 0$ and large $x_2$)
to nearly $+4$ (someone with $x_1 > 0$ and large negative $x_2$). A paper that reports only the ATE
would average away the fact that half the population gets a small or even harmful treatment.

## 2. The naive baseline: a single OLS estimate of the ATE

In [ ]:
ate_naive = sm.OLS(df["Y"].values,
                   sm.add_constant(df[["D"]].values)).fit().params[1]
print(f"OLS ATE (no covariates): {ate_naive:0.3f}")
print(f"OLS ATE (with X): "
      f"{sm.OLS(df['Y'].values, sm.add_constant(df[['D','x1','x2','x3','x4']].values)).fit().params[1]:0.3f}")
print(f"true ATE:           {df['tau_true'].mean():0.3f}")

The OLS ATE recovers the true average well. The question is what is hidden underneath that
average - the answer requires a CATE estimator.

## 3. A DML-style residual-on-residual approach to CATE

Double machine learning (DML) gives a clean way to estimate the CATE without contaminating the
treatment effect with baseline outcome variation. The idea, in three lines:

1. Estimate $\widehat m(\mathbf{x}) = \mathbb{E}[Y \mid \mathbf{x}]$ on a held-out fold.
2. Estimate $\widehat e(\mathbf{x}) = \mathbb{E}[D \mid \mathbf{x}]$ on the same held-out fold.
3. Form residuals $\tilde Y = Y - \widehat m$ and $\tilde D = D - \widehat e$. Regress $\tilde Y$ on
   $\tilde D$ in flexible interactions with $\mathbf{x}$. The coefficient that multiplies $\tilde D$
   is the CATE.

When treatment is randomized (as here), $\widehat e \approx 0.5$ for everyone and the residualization
of $D$ is innocuous. The residualization of $Y$ still matters: subtracting $\widehat m$ reduces
noise.

In [ ]:
def fit_dml_residuals(df, n_splits=5, seed=11):
    # Cross-fitted nuisance estimates. Returns Y-tilde, D-tilde, X.
    X = df[["x1","x2","x3","x4"]].values
    Y = df["Y"].values
    D = df["D"].values
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    m_hat = np.zeros_like(Y, dtype=float)
    e_hat = np.zeros_like(Y, dtype=float)
    for tr, te in kf.split(X):
        rf_y = RandomForestRegressor(n_estimators=200, min_samples_leaf=20,
                                     random_state=seed).fit(X[tr], Y[tr])
        rf_d = RandomForestRegressor(n_estimators=200, min_samples_leaf=20,
                                     random_state=seed).fit(X[tr], D[tr])
        m_hat[te] = rf_y.predict(X[te])
        e_hat[te] = rf_d.predict(X[te])
    Y_tilde = Y - m_hat
    D_tilde = D - e_hat
    return Y_tilde, D_tilde, X

Y_tilde, D_tilde, X = fit_dml_residuals(df)
ate_dml = (Y_tilde * D_tilde).sum() / (D_tilde ** 2).sum()
print(f"DML ATE estimate: {ate_dml:0.3f}")
print(f"true ATE:         {df['tau_true'].mean():0.3f}")

**Read the printout.** The DML ATE matches the OLS ATE because the treatment is randomized, but
the DML residuals are what we need for the heterogeneity step. Now we fit a CATE function. The honest
approach is to **split the residual sample** (Athey-Imbens 2016): use one half to grow trees that pick
splits, the other half to compute leaf estimates. Random forests built on the held-out half then
average those leaf estimates.

## 4. Hand-rolled honest causal forest

The recipe for a single honest tree:

1. Split the residual sample into halves $\mathcal{S}_{split}$ and $\mathcal{S}_{est}$.
2. On $\mathcal{S}_{split}$, build a tree that predicts the *pseudo-outcome* $\tilde Y / \tilde D$ from
   $\mathbf{x}$. The tree's splits become candidate cells with locally constant treatment effects.
3. On $\mathcal{S}_{est}$, drop each unit into its leaf and compute the local IV estimator
   $\widehat\tau_\ell = \sum_{i \in \ell} \tilde Y_i \tilde D_i / \sum_{i \in \ell} \tilde D_i^2$.

A causal forest averages this construction over many bootstrap trees. We grow $B = 60$ trees
(small for speed - 500-1000 in a real paper). For each test point $\mathbf{x}_0$, the forest
prediction is the average over trees of the leaf estimate that $\mathbf{x}_0$ falls into.

In [ ]:
def honest_causal_forest(X, Y_tilde, D_tilde, B=60, min_leaf=40, seed=13):
    n = len(Y_tilde)
    n_features = X.shape[1]
    forest_rng = np.random.default_rng(seed)
    trees = []      # list of (fitted tree, leaf->tau dict)
    for b in range(B):
        # subsample + split into S_split, S_est
        idx = forest_rng.choice(n, size=int(0.8 * n), replace=False)
        half = len(idx) // 2
        s_split = idx[:half]
        s_est   = idx[half:]
        # tree on S_split: split on X to predict pseudo-outcome
        # pseudo outcome uses small ridge to avoid divide-by-zero in tiny D_tilde
        pseudo = Y_tilde[s_split] * D_tilde[s_split] / (D_tilde[s_split] ** 2 + 1e-6)
        tree = DecisionTreeRegressor(min_samples_leaf=min_leaf,
                                     max_features="sqrt",
                                     random_state=int(forest_rng.integers(1e9))).fit(
            X[s_split], pseudo)
        # leaf-level tau on S_est
        leaves_est = tree.apply(X[s_est])
        tau_per_leaf = {}
        for leaf in np.unique(leaves_est):
            mask = leaves_est == leaf
            num = (Y_tilde[s_est][mask] * D_tilde[s_est][mask]).sum()
            den = (D_tilde[s_est][mask] ** 2).sum()
            tau_per_leaf[leaf] = num / den if den > 1e-6 else 0.0
        trees.append((tree, tau_per_leaf))
    return trees

def forest_predict(trees, X_new):
    preds = np.zeros(len(X_new))
    counts = np.zeros(len(X_new))
    for tree, tau_per_leaf in trees:
        leaves = tree.apply(X_new)
        for i, leaf in enumerate(leaves):
            if leaf in tau_per_leaf:
                preds[i] += tau_per_leaf[leaf]
                counts[i] += 1
    return preds / np.maximum(counts, 1)

trees = honest_causal_forest(X, Y_tilde, D_tilde, B=60, min_leaf=40)
tau_hat = forest_predict(trees, X)
print(f"forest produced {len(trees)} trees")
print(f"correlation of tau_hat with truth: {np.corrcoef(tau_hat, df['tau_true'].values)[0,1]:0.3f}")
print(f"MSE: {np.mean((tau_hat - df['tau_true'].values)**2):0.3f}")

**What that correlation means.** A coefficient near $0.6-0.8$ is realistic for a small forest on
$N=2000$ with two real effect modifiers and two noise covariates. A real-world CausalForestDML on the
same DGP would do a touch better thanks to honest forests' theoretical guarantees and bigger $B$, but
the qualitative pattern is identical: the forest **discovers** that $x_1$ and $x_2$ matter without
being told.

## 5. Looking at the heterogeneity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df["x1"], tau_hat, s=6, alpha=0.4, label="estimated $\\hat\\tau$")
axes[0].scatter(df["x1"], df["tau_true"], s=6, alpha=0.4, color="C3", label="truth")
axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("treatment effect"); axes[0].legend()
axes[0].set_title("CATE vs $x_1$ (real modifier)")
axes[1].scatter(df["x2"], tau_hat, s=6, alpha=0.4, label="$\\hat\\tau$")
axes[1].scatter(df["x2"], df["tau_true"], s=6, alpha=0.4, color="C3", label="truth")
axes[1].set_xlabel("$x_2$"); axes[1].set_title("CATE vs $x_2$ (real modifier)")
axes[1].legend()
plt.tight_layout(); plt.close(fig)
"plotted (figure closed)"


In [ ]:
# noise covariate: should look flat
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["x3"], tau_hat, s=6, alpha=0.4, label="$\\hat\\tau$")
ax.scatter(df["x3"], df["tau_true"], s=6, alpha=0.4, color="C3", label="truth")
ax.set_xlabel("$x_3$"); ax.set_ylabel("treatment effect")
ax.set_title("CATE vs $x_3$ (noise covariate, should be flat)")
ax.legend()
plt.tight_layout(); plt.close(fig)
"plotted"


**Read the plots.** Along $x_1$ the forest learns a step pattern: low effect for $x_1 \le 0$,
high effect for $x_1 > 0$. Along $x_2$ it learns a downward slope. Along $x_3$ the forest produces
a near-flat line - it correctly ignores the noise covariate. If you saw a real slope on $x_3$ you would
worry about overfitting.

## 6. Best linear projection: a publishable summary

A forest is a great prediction object but a lousy paragraph. The standard reporting device is the
**best linear projection (BLP)** of $\widehat\tau(\mathbf{x})$ onto a small set of summary variables:
$$\widehat\tau(\mathbf{x}) \approx \beta_0 + \beta_1 g_1(\mathbf{x}) + \cdots + \beta_p g_p(\mathbf{x}),$$
with $g_j$ chosen for interpretability (a few principal components, dummies for tertiles of the most
important features, etc.). The recipe of Semenova & Chernozhukov (2021): regress
$\widehat\tau(\mathbf{x})$ on $g(\mathbf{x})$ via OLS, using sample-splitting to keep inference valid.
We will project onto $g(\mathbf{x}) = (\mathbb{1}\{x_1 > 0\}, x_2)$, the two variables we *suspect* drive
heterogeneity. If the BLP slopes match the DGP coefficients, the forest is telling a coherent story.

In [ ]:
g1 = (df["x1"].values > 0).astype(float)
g2 = df["x2"].values
G = np.column_stack([g1, g2])
blp = sm.OLS(tau_hat, sm.add_constant(G)).fit(cov_type="HC1")
print("Best linear projection of CATE onto (1{x1>0}, x2):")
print(blp.summary().tables[1])
print()
print("DGP truth says: tau(x) = 1.0 + 1.5*1{x1>0} - 0.8*x2")
print("BLP            : intercept,  slope on 1{x1>0},  slope on x2")
print(f"               : {blp.params[0]:0.3f},        {blp.params[1]:0.3f},               {blp.params[2]:0.3f}")

**The reveal.** The BLP slopes are not the *exact* DGP coefficients (the forest is biased toward
the centre of the covariate space), but they are close, and - crucially - their HC1 t-statistics are
big enough that a referee would believe the heterogeneity is real. The right way to *report*
heterogeneity in a paper is exactly this kind of three-line BLP table, not a wall of leaf-level numbers.

## 7. A pre-registration template you can copy

Heterogeneity hunting is where p-hacking lives. The discipline that protects you in revision is to
pre-register the heterogeneity dimensions before you peek. The minimum viable template:

> **Heterogeneity pre-registration.** We will estimate the conditional average treatment effect via a
> causal forest with honest splitting (Athey & Wager 2019). The forest will use $B = 1000$ trees, a
> minimum leaf size of 30, and a $50/50$ honest split. We will summarize heterogeneity via the best
> linear projection (Semenova & Chernozhukov 2021) onto the following pre-specified variables:
> [list 3-6 variables]. We will report the BLP table with HC1 standard errors. We will also report a
> single-cut visualization of CATE versus each pre-specified moderator. We will *not* drop covariates
> from the BLP after seeing the results.

The crucial line is the last one. Without it, every robustness check you run after the fact becomes a
new degree of freedom and you slide into the multiple-testing problem of nb10.1.

## 8. When causal forests fail (be honest)

1. **Sparse covariates.** A forest needs enough data in each leaf to estimate a treatment effect.
   With $N < 500$ or with very heavy effect modifiers, leaves get tiny and the standard errors
   explode.
2. **Non-overlap.** If propensity $e(\mathbf{x})$ is near 0 or 1 in some regions of $\mathbf{x}$,
   the local IV inversion in the leaf estimator blows up. Trim units with extreme propensity, or
   restrict the population.
3. **Bad nuisance.** If $\widehat m$ or $\widehat e$ overfit, the residuals are not orthogonal and
   the DML residual-on-residual logic breaks. Cross-fitting (the `KFold` step in section 3) is the
   defence; use it.
4. **Honesty matters more than you think.** A causal forest *without* honest splitting will produce
   tighter-looking standard errors that lie about coverage. Athey-Imbens 2016 is the canonical
   reference.

## 9. Your turn

1. **Knock out an effect modifier.** Re-run the DGP with the step in $x_1$ removed (drop the
   `+1.5*(X[:,0]>0)` term). Re-run the forest. Does it correctly produce a near-flat $\widehat\tau$
   along $x_1$ now?
2. **Add overlap pain.** Modify the propensity to $e(\mathbf{x}) = \Phi(2 x_1)$. Treatment is no
   longer randomized. Recompute DML; what happens to the ATE if you skip the propensity model?
3. **Pre-register vs. peek.** Generate the DGP without looking, then write three sentences describing
   which covariates you *guess* drive heterogeneity. Now estimate; how many guesses survived?

**Citations.** Athey & Imbens (2016, *PNAS* 113(27):7353-7360); Athey, Tibshirani & Wager (2019,
*Annals of Statistics* 47(2):1148-1178); Wager & Athey (2018, *JASA* 113(523):1228-1242);
Chernozhukov, Chetverikov, Demirer, Duflo, Hansen, Newey & Robins (2018, *Econometrics Journal*
21(1):C1-C68); Semenova & Chernozhukov (2021, *Econometrics Journal* 24(2):264-289).